# 03 — Análise RFM e Segmentação de Clientes

Cálculo das métricas de Recência, Frequência e Valor Monetário (RFM)
para segmentação comportamental dos clientes.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/processed/online_retail_cleaned.csv')

In [3]:
df['TotalPrice'] = df['Quantity'] * df['Price']
# Criação da coluna Total Price

In [4]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00
...,...,...,...,...,...,...,...,...,...
779490,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60
779491,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60
779492,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60
779493,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,14.85


In [5]:
df.dtypes

Invoice          int64
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID      int64
Country         object
TotalPrice     float64
dtype: object

In [6]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [7]:
data_referencia = df['InvoiceDate'].max() + pd.Timedelta(days=1)
data_referencia

Timestamp('2011-12-10 12:50:00')

Cálculo da Recência

In [8]:
rfm = df.groupby('Customer ID').agg(
    recencia = ('InvoiceDate', lambda x: (data_referencia - x.max()).days),
    frequencia = ('Invoice', 'nunique'),
    monetario = ('TotalPrice', 'sum')
).reset_index()

In [9]:
rfm.head()

,Customer ID,recencia,frequencia,monetario
0,12346,326,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


In [10]:
print(rfm.describe())

        Customer ID     recencia   frequencia      monetario
count   5881.000000  5881.000000  5881.000000    5881.000000
mean   15314.674205   201.457745     6.287196    2954.396237
std     1715.429759   209.474135    13.012879   14437.322635
min    12346.000000     1.000000     1.000000       0.000000
25%    13833.000000    26.000000     1.000000     341.900000
50%    15313.000000    96.000000     3.000000     865.600000
75%    16797.000000   380.000000     7.000000    2247.720000
max    18287.000000   739.000000   398.000000  580987.040000


In [11]:
rfm = rfm[rfm['monetario'] > 0]

In [12]:
rfm.shape

(5878, 4)

In [13]:
rfm['R'] = pd.qcut(rfm['recencia'], q=5, labels=[5, 4, 3, 2, 1])
rfm['F'] = pd.qcut(rfm['frequencia'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M'] = pd.qcut(rfm['monetario'], q=5, labels=[1, 2, 3, 4, 5])

print(rfm.head())

   Customer ID  recencia  frequencia  monetario  R  F  M
0        12346       326          12   77556.46  2  5  5
1        12347         2           8    4921.53  5  4  5
2        12348        75           5    2019.40  3  4  4
3        12349        19           4    4428.69  5  3  5
4        12350       310           1     334.40  2  1  2


In [14]:
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

In [15]:
print(rfm.head())

   Customer ID  recencia  frequencia  monetario  R  F  M RFM_Score
0        12346       326          12   77556.46  2  5  5       255
1        12347         2           8    4921.53  5  4  5       545
2        12348        75           5    2019.40  3  4  4       344
3        12349        19           4    4428.69  5  3  5       535
4        12350       310           1     334.40  2  1  2       212


Criação da função para segmentar os clientes

In [17]:
def segmentar(row):
    r = int(row['R'])
    f = int(row['F'])

    if r >= 4 and f >= 4:
        return 'Campeões'
    elif r >= 3 and f >= 3:
        return 'Clientes Fiéis'
    elif r >= 4 and f <= 2:
        return 'Novos Clientes'
    elif r >= 3 and f <= 2:
        return 'Potenciais Fiéis'
    elif r <= 2 and f >= 4:
        return 'Não Posso Perder'
    elif r <= 2 and f >= 3:
        return 'Em Risco'
    else:
        return 'Perdidos'

rfm['Segmento'] = rfm.apply(segmentar, axis=1)
print(rfm['Segmento'].value_counts())

Segmento
Perdidos            1523
Campeões            1482
Clientes Fiéis      1221
Em Risco             471
Novos Clientes       443
Potenciais Fiéis     385
Não Posso Perder     353
Name: count, dtype: int64


In [18]:
rfm.to_csv('../data/processed/rfm.csv', index=False)

### Principais insights obtidos:

- 25% da base são Clientes Campeões, ou seja, compraram recentemente, com alta frequência e alto valor.
- Clientes perdidos são o maior segmento, o que significa que temos um problema de retenção.
- 8% dos clientes estão em risco, significa que já foram bons compradores estão deixando de comprar.